# `pathlib` w pipeline'ie danych — pełny przewodnik

**Problem:** operacje na ścieżkach plików (`os.path.join`, `os.listdir`, ręczne sklejanie stringów z `/`) są rozproszone po wielu funkcjach modułu `os`, niespójne między Windows/Linux, i traktują ścieżkę jako zwykły tekst. `pathlib` reprezentuje ścieżkę jako **obiekt** z metodami — czytelniejszy kod, mniej błędów przy przenoszeniu między systemami, i bezpośrednia integracja z pandas (funkcje takie jak `pd.read_csv` przyjmują obiekt `Path` tak samo jak string).

**Porównanie:**
- `os.path.join("data", "raw", "plik.csv")` — sklejanie stringów, wynik to zwykły `str`.
- `Path("data") / "raw" / "plik.csv"` — budowanie obiektu przez operator `/`, wynik ma metody (`.exists()`, `.stem`, `.parent`...).

**Zakres notatki:** podstawy obiektu `Path`, budowanie i rozbieranie ścieżek, wyszukiwanie plików (`glob`/`rglob`), odczyt/zapis (w tym integracja z pandas), oraz gotowe receptury pod typowy pipeline: pobranie → transformacja → zapis.

## Setup

Budujemy przykładową strukturę projektu z kilkoma plikami CSV — będzie towarzyszyć całej notatce.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

rng = np.random.default_rng(3)

project = Path("project_demo")
for sub in ["data/raw", "data/processed", "data/output", "logs"]:
    (project / sub).mkdir(parents=True, exist_ok=True)

raw_dir = project / "data/raw"
for month in ["styczen", "luty", "marzec"]:
    df = pd.DataFrame({
        "region": rng.choice(["North", "South", "East"], 5),
        "sales": rng.integers(500, 2000, 5),
    })
    df.to_csv(raw_dir / f"sales_{month}.csv", index=False)

(raw_dir / "notes.txt").write_text("uwagi do danych źródłowych")

print("Struktura utworzona.")

## Sekcja 1 — Podstawy: tworzenie obiektu `Path`

In [ ]:
p = Path("data/raw/sales_2026.csv")
print(p, type(p).__name__)

print(f"Bieżący katalog roboczy: {Path.cwd()}")
print(f"Katalog domowy: {Path.home()}")

## Sekcja 2 — Budowanie i rozbieranie ścieżek

Operator `/` sklejа segmenty ścieżki niezależnie od systemu operacyjnego — nie trzeba pamiętać, czy separator to `/` czy `\`.

In [ ]:
full_path = Path("data") / "raw" / "sales_2026.csv"
full_path

### Rozbieranie ścieżki na części

- `.name` — pełna nazwa pliku (z rozszerzeniem).
- `.stem` — nazwa BEZ rozszerzenia.
- `.suffix` — TYLKO OSTATNIE rozszerzenie (patrz `.suffixes` niżej dla wielokrotnych).
- `.parent` — katalog nadrzędny (jeden poziom wyżej).
- `.parents[n]` — n-ty poziom wyżej.
- `.parts` — cała ścieżka jako krotka segmentów.

In [ ]:
example = Path("/home/claude/data/raw/sales_2026.csv")

print(f"name:        {example.name}")
print(f"stem:        {example.stem}")
print(f"suffix:      {example.suffix}")
print(f"parent:      {example.parent}")
print(f"parents[1]:  {example.parents[1]}")
print(f"parts:       {example.parts}")

### `.suffix` vs `.suffixes` — pliki z wieloma rozszerzeniami

`.suffix` zwraca TYLKO ostatnie rozszerzenie — dla `archive.tar.gz` to `.gz`, nie `.tar.gz`. Jeśli potrzebujesz pełnego, złożonego rozszerzenia, użyj `.suffixes` (lista wszystkich).

In [ ]:
archive = Path("archive.tar.gz")
print(f"suffix:   {archive.suffix}")
print(f"suffixes: {archive.suffixes}")

## Sekcja 3 — Sprawdzanie: `exists`, `is_file`, `is_dir`, `stat`

In [ ]:
sample_file = raw_dir / "sales_styczen.csv"

print(f"Istnieje:     {sample_file.exists()}")
print(f"To plik:      {sample_file.is_file()}")
print(f"To katalog:   {sample_file.is_dir()}")

info = sample_file.stat()
print(f"Rozmiar:      {info.st_size} bajtów")
print(f"Czas zmiany:  {info.st_mtime} (epoch, do formatowania przez datetime.fromtimestamp)")

## Sekcja 4 — Tworzenie katalogów: `mkdir(parents=True, exist_ok=True)`

`parents=True` tworzy WSZYSTKIE brakujące katalogi nadrzędne naraz (jak `mkdir -p`). `exist_ok=True` sprawia, że wywołanie na już istniejącym katalogu nie rzuca błędu — kluczowe w skryptach uruchamianych wielokrotnie (np. w pipeline odpalanym codziennie).

In [ ]:
new_dir = project / "data/archive/2026"
new_dir.mkdir(parents=True, exist_ok=True)
print(f"Utworzono: {new_dir}")

# Ponowne wywołanie - BEZ błędu dzięki exist_ok=True
new_dir.mkdir(parents=True, exist_ok=True)
print("Ponowne wywołanie - bez błędu")

## Sekcja 5 — Wyszukiwanie plików: `iterdir`, `glob`, `rglob`

- `.iterdir()` — wszystkie wpisy w JEDNYM katalogu (bez wzorca, bez rekurencji).
- `.glob(wzorzec)` — dopasowanie wzorca (np. `"*.csv"`) w JEDNYM katalogu.
- `.rglob(wzorzec)` — to samo, ale REKURENCYJNIE przez wszystkie podkatalogi (odpowiednik `glob("**/wzorzec")`).

In [ ]:
print("glob('sales_*.csv') - tylko w raw_dir, dopasowane do wzorca:")
for f in sorted(raw_dir.glob("sales_*.csv")):
    print(f" ", f.name)

print("\nrglob('*.csv') - rekurencyjnie, przez CAŁY projekt:")
for f in sorted(project.rglob("*.csv")):
    print(" ", f)

**Uwaga:** `rglob("*")` (bez konkretnego rozszerzenia) zwraca WSZYSTKO — pliki I katalogi. Jeśli potrzebujesz tylko plików, dofiltruj `.is_file()`.

In [ ]:
all_entries = list(project.rglob("*"))
files_only = [f for f in all_entries if f.is_file()]
print(f"Wszystkie wpisy: {len(all_entries)}, tylko pliki: {len(files_only)}")

## Sekcja 6 — Odczyt i zapis

`.read_text()`/`.write_text()` do prostych plików tekstowych, bez ręcznego `open()`/`close()`. Kluczowe dla pipeline'u: **pandas przyjmuje obiekt `Path` bezpośrednio** — nie trzeba konwertować przez `str()`.

In [ ]:
notes_content = (raw_dir / "notes.txt").read_text()
print(f"Zawartość notes.txt: {notes_content!r}")

# pandas akceptuje Path bez konwersji do str
df = pd.read_csv(raw_dir / "sales_styczen.csv")
print(f"\nWczytano {len(df)} wierszy bezpośrednio z obiektu Path")

## Sekcja 7 — Receptura: wsadowe wczytanie wielu plików w jeden `DataFrame`

Klasyczny pierwszy krok pipeline'u: znajdź wszystkie pliki pasujące do wzorca, wczytaj każdy, doklej informację o źródle, połącz w jedną tabelę.

In [ ]:
all_files = sorted(raw_dir.glob("sales_*.csv"))

combined = pd.concat(
    [pd.read_csv(f).assign(source_file=f.name) for f in all_files],
    ignore_index=True,
)
combined

## Sekcja 8 — Budowanie ścieżek WYNIKOWYCH: `with_suffix`, `with_stem`, `with_name`

Te trzy metody NIE modyfikują istniejącego pliku — zwracają NOWY obiekt `Path` z jednym elementem podmienionym. Podstawa budowania ścieżki wyjściowej pipeline'u na podstawie ścieżki wejściowej.

In [ ]:
input_path = raw_dir / "sales_styczen.csv"

print(f"Zmiana rozszerzenia:        {input_path.with_suffix('.parquet')}")
print(f"Zmiana nazwy (bez rozsz.):  {input_path.with_stem(input_path.stem + '_processed')}")
print(f"Zmiana całej nazwy pliku:   {input_path.with_name('inny_plik.csv')}")

### Połączenie: inny katalog + inne rozszerzenie naraz

Typowy przypadek pipeline'u: dane wejściowe leżą w `raw/`, wynik ma trafić do `processed/` z innym rozszerzeniem.

In [ ]:
processed_dir = project / "data/processed"
output_path = processed_dir / input_path.with_suffix(".parquet").name
output_path

## Sekcja 9 — Zapis wyników

Tak jak przy odczycie, `to_parquet`/`to_csv` przyjmują obiekt `Path` bez konwersji.

In [ ]:
df.to_parquet(output_path)
print(f"Zapisano: {output_path}")
print(f"Rozmiar pliku: {output_path.stat().st_size} bajtów")

## Sekcja 10 — Receptura: najnowszy plik w katalogu

Częsty scenariusz: folder z cyklicznymi eksportami (np. codzienny zrzut z systemu źródłowego), a pipeline ma przetworzyć zawsze TEN NAJNOWSZY.

In [ ]:
import time

# Symulacja: dwa kolejne pliki z odstępem czasowym
for month in ["kwiecien", "maj"]:
    (raw_dir / f"sales_{month}.csv").write_text("region,sales\nNorth,100\n")
    time.sleep(0.01)

latest_file = max(raw_dir.glob("sales_*.csv"), key=lambda f: f.stat().st_mtime)
print(f"Najnowszy plik: {latest_file.name}")

## Sekcja 11 — Usuwanie: `unlink`, `rmdir`

In [ ]:
temp_file = raw_dir / "temp.txt"
temp_file.write_text("tymczasowy")
print(f"Istnieje przed: {temp_file.exists()}")

temp_file.unlink()
print(f"Istnieje po unlink(): {temp_file.exists()}")

# missing_ok=True - bez błędu, gdy pliku i tak już nie ma (bezpieczne w skryptach "sprzątających")
temp_file.unlink(missing_ok=True)
print("Ponowny unlink(missing_ok=True) - bez błędu")

**Uwaga:** `.rmdir()` usuwa TYLKO pusty katalog — na niepustym rzuca błąd (celowe zabezpieczenie przed przypadkową utratą danych). Do usunięcia całego drzewa katalogów z zawartością potrzebny jest `shutil.rmtree()` — `pathlib` świadomie tego nie oferuje wprost.

In [ ]:
empty_dir = project / "empty_test"
empty_dir.mkdir()
empty_dir.rmdir()
print("Pusty katalog usunięty bez problemu")

try:
    (project / "data").rmdir()
except OSError as e:
    print(f"Katalog niepusty -> błąd: {e}")

## Sekcja 12 — Pełne receptury pipeline

### Receptura A — Automatyczna struktura projektu na starcie skryptu

Jedna funkcja wywoływana na początku każdego uruchomienia pipeline'u — bezpieczna do wielokrotnego wywołania dzięki `exist_ok=True`.

In [ ]:
def ensure_project_structure(base: Path) -> dict[str, Path]:
    """Tworzy standardową strukturę katalogów pipeline'u i zwraca słownik ścieżek."""
    structure = {
        "raw": base / "data/raw",
        "processed": base / "data/processed",
        "output": base / "data/output",
        "logs": base / "logs",
    }
    for path in structure.values():
        path.mkdir(parents=True, exist_ok=True)
    return structure


paths = ensure_project_structure(project)
paths

### Receptura B — Pełny pipeline: pobranie → transformacja → zapis

Złożenie wszystkiego z tej notatki w jeden przepływ: znajdź pliki źródłowe, wczytaj i połącz, przekształć, zapisz z nazwą pochodną od źródła.

In [ ]:
def run_pipeline(paths: dict[str, Path]) -> Path:
    # 1. Pobranie - wszystkie pliki źródłowe pasujące do wzorca
    source_files = sorted(paths["raw"].glob("sales_*.csv"))

    # 2. Wczytanie i połączenie
    combined = pd.concat([pd.read_csv(f) for f in source_files], ignore_index=True)

    # 3. Transformacja (przykładowa - suma sprzedaży per region)
    result = combined.groupby("region", as_index=False)["sales"].sum()

    # 4. Zapis - nazwa wynikowa niezależna od tego, ile plików wejściowych przetworzono
    output_file = paths["output"] / "sales_summary.parquet"
    result.to_parquet(output_file)

    return output_file


result_path = run_pipeline(paths)
print(f"Pipeline zakończony. Wynik: {result_path}")
pd.read_parquet(result_path)

### Receptura C — Nazwa pliku loga z znacznikiem czasu

Przydatne przy pipeline uruchamianym cyklicznie (harmonogram zadań) — każde uruchomienie dostaje własny plik loga, nic się nie nadpisuje.

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
log_path = paths["logs"] / f"pipeline_run_{timestamp}.log"
log_path.write_text(f"Pipeline uruchomiony: {timestamp}\nPrzetworzono plików: 3\n")
print(f"Log zapisany: {log_path}")

## Sekcja 13 — Pułapki

### Pułapka 1 — `Path(__file__)` nie działa w Jupyter/notebookach

`__file__` to zmienna dostępna tylko przy uruchamianiu prawdziwego pliku `.py` — w interaktywnym kernelu Jupyter (jak ten notebook) w ogóle nie istnieje. Zweryfikowane bezpośrednio w tym środowisku poniżej. Typowe rozwiązanie: `Path.cwd()` w notebooku, `Path(__file__).parent` w skrypcie `.py`.

In [ ]:
try:
    Path(__file__)
except NameError as e:
    print(f"Błąd w notebooku: {e}")

print(f"Działający odpowiednik w notebooku: {Path.cwd()}")

### Pułapka 2 — kolejność wyników `glob()`/`rglob()` NIE jest gwarantowana

Kolejność zależy od systemu plików i nie jest częścią kontraktu API — na jednym systemie pliki mogą wyjść alfabetycznie, na innym w kolejności utworzenia. Jeśli kolejność przetwarzania ma znaczenie (np. pliki miesięczne muszą być połączone chronologicznie), **zawsze** owijaj wynik w `sorted()` — tak jak robiliśmy to konsekwentnie w każdym przykładzie tej notatki.

### Pułapka 3 — `rmdir()` działa tylko na pustym katalogu

Opisane w Sekcji 11 — celowe zabezpieczenie, nie błąd biblioteki. Do usunięcia niepustego drzewa katalogów potrzebny jest `shutil.rmtree()`, spoza `pathlib`.

## Podsumowanie

| Zadanie | Metoda/atrybut |
|---|---|
| Budowanie ścieżki niezależnie od OS | `Path("a") / "b" / "c"` |
| Nazwa pliku / bez rozszerzenia / rozszerzenie | `.name` / `.stem` / `.suffix` |
| Wielokrotne rozszerzenia (`.tar.gz`) | `.suffixes` |
| Katalog nadrzędny / n poziomów wyżej | `.parent` / `.parents[n]` |
| Czy plik/katalog istnieje | `.exists()` / `.is_file()` / `.is_dir()` |
| Rozmiar / czas modyfikacji | `.stat().st_size` / `.stat().st_mtime` |
| Utworzenie zagnieżdżonych katalogów bezpiecznie | `.mkdir(parents=True, exist_ok=True)` |
| Pliki wg wzorca w jednym katalogu | `.glob("*.csv")` |
| Pliki wg wzorca rekurencyjnie | `.rglob("*.csv")` |
| Odczyt/zapis prostego tekstu | `.read_text()` / `.write_text()` |
| Wczytanie danych przez pandas | `pd.read_csv(sciezka)` — `Path` bezpośrednio, bez `str()` |
| Zmiana rozszerzenia ścieżki wynikowej | `.with_suffix(".parquet")` |
| Zmiana nazwy z zachowaniem rozszerzenia | `.with_stem(nowa_nazwa)` |
| Zmiana całej nazwy pliku | `.with_name("inny.csv")` |
| Najnowszy plik w katalogu | `max(sciezka.glob(...), key=lambda f: f.stat().st_mtime)` |
| Usunięcie pliku (bezpiecznie) | `.unlink(missing_ok=True)` |
| Usunięcie pustego katalogu | `.rmdir()` (niepusty → błąd, celowo) |
| Katalog roboczy zamiast `__file__` w notebooku | `Path.cwd()` |

**Wniosek:** `pathlib` zyskuje najwięcej w dokładnie tym miejscu pipeline'u, gdzie w `os.path` trzeba by pamiętać kolejność argumentów kilku różnych funkcji (`os.path.join`, `os.path.splitext`, `os.makedirs`) — tu to spójne metody jednego obiektu, które łączą się w łańcuch (`input_path.with_suffix(".parquet").name`) i działają identycznie niezależnie od systemu operacyjnego.